# BB Scattering Analysis: Part 2

In this notebook, we will take a second pass at the analysis for your BB scattering data. This time however, we will consider the uncertainty in the data and see how that affects our analysis technique and, ultimately, our results. 

Let's reset your analysis in the cell below. Import numpy and matplotlib. Reassign the values you had in your last notebook (`dx`, `h`, `num_events_tot`, etc.).

Then, plot your data to make sure that there are no errors in your script. For simplicity, everyone should use the `N` as a function of `sin(theta/2)` figure for today's analysis.

In [ ]:
# Your code goes here.

## Y-errorbars

The y-axis is counting the number of events. For counting experiments, the uncertainty on $N$ events is $\sqrt{N}$. *If you'd like to learn more about why this is, talk to your instructor or take PHY 322: Data Analysis & Visualization!*

In the cell below, create an array of values for the uncertainty in `num_events_tot`. Label that array `d_num_events_tot`. Then plot those results using the `errorbar` function.

In [ ]:
d_num_events_tot = ?

fig = plt.figure()
ax = fig.add_subplot(111)
ax.errorbar(?, ?, yerr=d_num_events_tot, fmt='o', capsize=3)

## X-errorbars

The x-axis is $\sin(\theta/2)$. We need to find the uncertainty in that value. The experimental values that went into the x-axis are:
- `r_detector`
- `dx`

To get a sense of the size of the error on those values, you would have to do repeated measurements to get a mean and standard deviation. If you made the initial measurement carefully, that's likely a relatively small error when compared to the size of the error along the y-axis. There might be additional errors in the measured angle due to how the tape was mounted and the location of $\theta=0$. Here, we are left to make an educated guess. For this analysis, we'll assume that the angle is precise within $\pm1$ degree, or $\pm0.0174$ rad.

You need to use the partial derivative method to calculate $\Delta\sin(\theta/2)$ from $\Delta\theta$. Do that calculation and assign the value to `d_sin_theta`. Then plot the data including both xerr and yerr. *Hint: you can specify an array for the xerr values similarly to how you did it for the yerr values.*

In [ ]:
# Your code goes here

## Weighted fits: an example

The errorbars on a data point are the measure of confidence that repeating the experiment would yield the same data point. This can come into play when we fit our data to know models to verify agreement, look for new trends, etc. You can force the fit to give more significance to data points with smaller errorbars by using algorithms that calculate **weighted fits**. 

To learn how to do this, let's consider an example of a different experiment. Imagine that you had measurements of position vs. time (as well as uncertainties on both quantities) for a projectile launched upwards and traveling for 2 seconds. The cells below download an example data file and plot the results:

In [ ]:
%%capture
!wget --no-check-certificate 'https://raw.githubusercontent.com/ryantrainor/PHY223/refs/heads/main/BB_Scattering/projectile-time-position.csv' -O projectile-time-position.csv


In [ ]:
# read the downloaded data file and plot the positions vs. time with error bars
projectile_data = np.loadtxt("projectile-time-position.csv", skiprows=1, delimiter=',')

time = projectile_data[:,0]
d_time = projectile_data[:,1]
position = projectile_data[:,2]
d_position = projectile_data[:,3]

fig = plt.figure()
ax = fig.add_subplot(111)
ax.errorbar(time, position, xerr=d_time, yerr=d_position, fmt='o', color='k', capsize=3)
ax.set_xlabel('time (sec)')
ax.set_ylabel('position (m)')

We have seen previously how to fit a function to unweighted data using the `curve_fit` function from `scipy.optimize`. With only small changes, the same function can be used to produce a fit that weights based on the uncertainties in the y direction as shown below:

In [ ]:
from scipy.optimize import curve_fit

# Fitting function for scipy curve_fit fitting
# Recall that the function must take an independent variable, followed by as 
# many additional variables as there are model parameters.
def freefall_func_scipy(t, x0, v0, acc):
  return x0 + v0*t + 0.5 * acc * t**2

init_params = [0., 0., 2.]

popt_ywt, pcov_ywt = curve_fit(freefall_func_scipy, time, position, p0=init_params,
                       sigma=d_position)
best_params_ywt = popt_ywt
best_param_uncs_ywt = np.sqrt(np.diag(pcov_ywt))

## Print the best-fit parameter values and their uncertainties
print("The best-fitted parameters are:")
print(f" init position = {best_params_ywt[0]:4.1f}\u00B1{best_param_uncs_ywt[0]:3.1f}")
print(f" init velocity = {best_params_ywt[1]:4.1f}\u00B1{best_param_uncs_ywt[1]:3.1f}")
print(f" acceleration  = {best_params_ywt[2]:4.1f}\u00B1{best_param_uncs_ywt[2]:3.1f}")


Note that we have specified the optional keyword argument `sigma` in the call to `curve_fit()`, which is what tells the the function to weight the points by their uncertainties in the fit. Specifically, each point $(x_i,\,y_i)$ receives a weight $1/\sigma_{y,i}^2$, where $\sigma_{y,i}$ is the uncertainty on the y-coordinate of the i$^\textrm{th}$ data point.

The cell below shows the resulting best-fit function based on this weighted fit. Complete the missing portion of the cell so that it also calculates an unweighted fit, and then run the cell to compare both fits to the data. *Hint: for the unweighted fit, you just need to remove a bit from the line used for the weighted fit.*

In [ ]:
# Calculate an unweighted ("unwt") fit to the data
popt_unwt, pcov_unwt = ?
best_params_unwt = ?
best_param_uncs_unwt = ?

## Plot the best-fit functions as well as the data

# create a densely-sampled time array for plotting the model
time_long = np.linspace(0, 2, 100)

# make the figure
fig = plt.figure()
ax = fig.add_subplot(111)
ax.errorbar(time, position, xerr=d_time, yerr=d_position, fmt='o', color='k',
            capsize=3, label='Measurements')
ax.plot(time_long, freefall_func_scipy(time_long, *best_params_ywt),color='r',
        label='SciPy fit (y-wt)')
ax.plot(time_long, freefall_func_scipy(time_long, *best_params_unwt),color='orange',
        label='SciPy fit (unwt)')
ax.legend()
ax.set_xlabel('time (sec)')
ax.set_ylabel('position (m)')

You should see above that the weighted fit falls farther from the points with large error bars (e.g., the point in the lower right), since these points have less weight for determining the parameters of the fit curve.

However, our points also have significant uncertainties in the x direction. SciPy does not have a native method for dealing with uncertainties in the x- and y-direction simultaneously, but one method to do so is called *orthogonal distance regression (ODR)*.

We can install the python library `odrpack` that provides tools for orthogonal distance regression using the code below:

In [ ]:
# %pip install odrpack

The cell below shows how to fit a function with the `odr_fit` function from `odrpack`. Note that `odr_fit` has some similarities to `curve_fit`, but pay attentino to the differences in the exact syntax below.

In [ ]:
from odrpack import odr_fit

# Fitting function for ODR fitting
# Note that this method requires the function to take an independent variable,
# followed by a single second variable that contains ALL the model parameters
def freefall_func_odr(t, beta):
    # note beta is an array that contains the three parameters:
    # initial position, initial velocity, and the acceleration
    x0, v0, acc = beta
    return x0 + v0*t + 0.5 * acc * t**2

init_params = [0., 0., 2.]

sol = odr_fit(freefall_func_odr, time, position, beta0=init_params,
              weight_x=1/d_time**2,
              weight_y=1/d_position**2)

best_params_odr = sol.beta
best_param_uncs_odr = sol.sd_beta

## Print the best-fit parameter values and their uncertainties
print("The best-fitted parameters are:")
print(f" init position = {best_params_odr[0]:5.1f}\u00B1{best_param_uncs_odr[0]:3.1f}")
print(f" init velocity = {best_params_odr[1]:5.1f}\u00B1{best_param_uncs_odr[1]:3.1f}")
print(f" acceleration  = {best_params_odr[2]:5.1f}\u00B1{best_param_uncs_odr[2]:3.1f}")



In particular, note the following differences compared to `curve_fit`:
1. `odr_fit` requires all the parameters to be fit in the model to be passed to the model function as a single array-like variable (beta).
2. `odr_fit` requires providing the weights in the x- and y-directions, rather than providing the uncertainties directly.
3. The output of `odr_fit` is a single object that contains both the best-fit parameters and their uncertainties as *attributes*, rather than returning an array of best-fit parameters and a covariance matrix.

You can find more details about the options for using `odr_fit` in the [odrpack documentation](https://hugomvale.github.io/odrpack-python/reference/). There are many other fitting methods available in Python as well, which will have other formats for their inputs and outputs; make sure to review the documentation for whatever method you are using for a given project.

Finally, we'll use the cell below to compare all three of the fits we created (unweighted, y-weighted, and ODR):

In [ ]:
## Plot both best-fit functions as well as the data
fig = plt.figure()
ax = fig.add_subplot(111)
ax.errorbar(time, position, xerr=d_time, yerr=d_position, fmt='o', color='k',
            capsize=3, label='Measurements')
ax.plot(time_long, freefall_func_scipy(time_long, *best_params_ywt),color='r',
        label='SciPy fit (y-wt)')
ax.plot(time_long, freefall_func_scipy(time_long, *best_params_unwt),color='orange',
        label='SciPy fit (unwt)')
ax.plot(time_long, freefall_func_odr(time_long, best_params_odr), color='b',
        label='ODR fit')
ax.legend()
ax.set_xlabel('time (sec)')
ax.set_ylabel('position (m)')

Again note the differences in the three fits. Pay particular attention to the differences between the ODR fit and the unweighted fit: for which points do the sensitivities of the two fits differ the most? Why? Be prepared to talk to your instructor about this in the next section.

## BB Scattering with a weighted fit

Now it's your turn: use the cells below to conduct a weighted fit to the data. Before you begin, discuss the following with your group members and then with your instructor:
1. Do you expect that your results will depend very much on whether you use a weighted vs. an unweighted fit? Why?
2. If you use a weighted fit, do you expect that results will depend very much on whether you use a y-weighted vs. an ODR fit? Why?
3. Choose one of the two weighted fit methods discussed above to repeat your analysis of the BB scattering data. which method will you use? Why?

Once your plan has been approved by your instructor, use the cell below to calculate a new estimate of $R$ and its uncertainty based on your BB scattering data:

In [ ]:
# Implement a weighted fit


Compare your results to the value you measured from the unweighted fit in Part 1 of the analysis. Is your estimate of the radius significantly different, or is it consistent? How do you know?

Now plot your new best-fit model along with your data. Make sure to include error bars, a legend, axes labels, and any other elements necessary to clearly communicate your results. *For maximum credit, also find a way to graphically indicate the range of fit curves allowed by your uncertainty in $R$ on your plot.*

In [ ]:
# Make a plot of your resulting best-fit model

You have now completed Part 2 of your analysis of the BB scattering experiment. If you have additional time, see if you can make additional improvements to the clarity of your plot. You may find the following resources useful:
- [Matplotlib documentation for the `errorbar()` function](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.errorbar.html)
- [Matplotlib documentation for `fill_between()`, a useful function for shading regions](https://matplotlib.org/stable/gallery/lines_bars_and_markers/fill_between_demo.html)
- [Matplotlib example gallery of lines, bar, and markers](https://matplotlib.org/stable/gallery/lines_bars_and_markers/index.html)
- [Matplotlib list of named colors](https://matplotlib.org/stable/gallery/color/named_colors.html)
- [Matplotlib tutorial on customizing tick marks](https://matplotlib.org/stable/users/explain/axes/axes_ticks.html)